In [165]:
import pandas as pd

In [166]:
tracking = pd.read_parquet('../data/tracking.parquet', engine="pyarrow")

In [183]:
teams = pd.read_csv("../results/teams.csv")
team_names = tracking.groupby('team_id').agg({'team_name':'max'}).reset_index()
teams = pd.merge(teams, team_names, how='left', on='team_id')
teams.to_csv('../results/teams.csv', index=False)

In [167]:
players = pd.read_csv('../results/players.csv')

In [168]:
players = players[['player_id', 'position_group']]
players = players[players['position_group'] != 'G']

In [169]:
oz_seqs = pd.read_csv('../results/full_entry_details.csv')
# oz_seqs = oz_seqs.rename(columns={'entry_player':'player_id', 'ozone_team':'team_id'})

In [170]:
game_ids = oz_seqs['game_id'].unique()
# subset_games = game_ids[0:5]

In [171]:
tracking = tracking[tracking['game_id'].isin(game_ids)]
tracking_players = pd.merge(tracking, players, on='player_id', how='inner')

In [172]:
oz_seq_game_events = oz_seqs[['game_id', 'sl_event_id']]

In [173]:
puck_carrier_tracking = pd.merge(oz_seqs, tracking, on=['game_id', 'sl_event_id', 'team_id', 'player_id'], how='left')
puck_carrier_tracking = puck_carrier_tracking.rename(columns={'tracking_x':'puck_carrier_x', 'tracking_y':'puck_carrier_y', 'tracking_vel_x': 'puck_carrier_vel_x', 'tracking_vel_y': 'puck_carrier_vel_y'})
puck_carrier_tracking['puck_carrier_x'], puck_carrier_tracking['puck_carrier_vel_x'] = puck_carrier_tracking['attack_direction'] * puck_carrier_tracking['puck_carrier_x'],  puck_carrier_tracking['attack_direction'] * puck_carrier_tracking['puck_carrier_vel_x']
puck_carrier_tracking['carrier_dist_from_blueline'] = puck_carrier_tracking['puck_carrier_x'] - 25

In [174]:
non_carrier_tracking = tracking.rename(columns={'player_id':'non_carrier_id'})
non_carrier_tracking = pd.merge(oz_seqs, non_carrier_tracking, on=['game_id', 'sl_event_id', 'team_id'], how="left")
non_carrier_tracking = non_carrier_tracking[non_carrier_tracking['player_id'] != non_carrier_tracking['non_carrier_id']]
non_carrier_tracking = non_carrier_tracking.rename(columns={'tracking_x':'non_carrier_x', 'tracking_y':'non_carrier_y', 'tracking_vel_x': 'non_carrier_vel_x', 'tracking_vel_y': 'non_carrier_vel_y'})
non_carrier_tracking = non_carrier_tracking[['game_id', 'sl_event_id', 'attack_direction', 'non_carrier_id', 'non_carrier_x', 'non_carrier_y', 'non_carrier_vel_x', 'non_carrier_vel_y']]
non_carrier_tracking['non_carrier_x'], non_carrier_tracking['non_carrier_vel_x'] = non_carrier_tracking['attack_direction'] * non_carrier_tracking['non_carrier_x'],  non_carrier_tracking['attack_direction'] * non_carrier_tracking['non_carrier_vel_x']
non_carrier_tracking['non_carrier_near_ozone'] = (non_carrier_tracking['non_carrier_x'] > 0).astype(int)
non_carrier_tracking['non_carrier_near_dzone'] = (non_carrier_tracking['non_carrier_x'].between(-25, 0)).astype(int)
non_carrier_tracking['non_carrier_in_dzone'] = (non_carrier_tracking['non_carrier_x'] < -25).astype(int)

In [175]:
defense_tracking = tracking.rename(columns={'player_id':'opponent_id', 'team_id':'opp_team_id', 'tracking_x':'opponent_x', 'tracking_y':'opponent_y', 'tracking_vel_x':'opponent_vel_x', 'tracking_vel_y':'opponent_vel_y'})
defense_tracking = pd.merge(oz_seqs, defense_tracking, on=['game_id', 'sl_event_id', 'opp_team_id'], how="left")
defense_tracking = defense_tracking[['game_id', 'sl_event_id', 'attack_direction', 'opponent_id', 'opponent_x', 'opponent_y', 'opponent_vel_x', 'opponent_vel_y']]
defense_tracking['opponent_x'], defense_tracking['opponent_vel_x'] = -1*defense_tracking['attack_direction'] * defense_tracking['opponent_x'],  -1*defense_tracking['attack_direction'] * defense_tracking['opponent_vel_x']
defense_tracking['opponent_near_ozone'] = (defense_tracking['opponent_x'] > 0).astype(int)
defense_tracking['opponent_near_dzone'] = (defense_tracking['opponent_x'].between(-25, 0)).astype(int)
defense_tracking['opponent_in_dzone'] = (defense_tracking['opponent_x'] < -25).astype(int)

In [176]:
non_carrier_tracking = non_carrier_tracking.groupby(['game_id', 'sl_event_id']).agg({'non_carrier_id':lambda x: list(x), 'non_carrier_x': lambda x: list(x), 'non_carrier_y': lambda x: list(x), 'non_carrier_vel_x':'median', 'non_carrier_vel_y':'median', 'non_carrier_near_ozone':'sum', 'non_carrier_near_dzone':'sum', 'non_carrier_in_dzone':'sum'}).reset_index()

In [177]:
defense_tracking = defense_tracking.groupby(['game_id', 'sl_event_id']).agg({'opponent_id':lambda x: list(x), 'opponent_x': lambda x: list(x), 'opponent_y': lambda x: list(x), 'opponent_vel_x':'median', 'opponent_vel_y':'median', 'opponent_near_ozone':'sum', 'opponent_near_dzone':'sum', 'opponent_in_dzone':'sum'}).reset_index()

In [178]:
print(non_carrier_tracking.head())

                                game_id  sl_event_id  \
0  00b0366a-95c6-5250-2dae-e3dd5c4198bc         28.0   
1  00b0366a-95c6-5250-2dae-e3dd5c4198bc         36.0   
2  00b0366a-95c6-5250-2dae-e3dd5c4198bc         48.0   
3  00b0366a-95c6-5250-2dae-e3dd5c4198bc         68.0   
4  00b0366a-95c6-5250-2dae-e3dd5c4198bc         92.0   

                                      non_carrier_id  \
0  [efbd8365-7b4a-13f4-6a00-d77d11d64910, dc5a9c1...   
1  [cf9158fc-30e9-680c-b9e5-b65717e1e000, a434fff...   
2  [8bd8e696-4514-a5d5-792b-80b002761f46, cf9158f...   
3  [cf9158fc-30e9-680c-b9e5-b65717e1e000, 8bd8e69...   
4  [e9ef1506-4a97-1586-e7b4-a1dfbec02e4e, None, N...   

                                       non_carrier_x  \
0     [-25.23622128, -4.00590564, 6.368110440000001]   
1  [24.38976456, 2.39829404, 15.360892880000002, ...   
2              [38.26443692, 43.307088, 16.78477744]   
3    [25.57086696, -4.66863532, -28.792651839999998]   
4  [8.667979279999999, -0.9678477999999999, 8.

In [179]:
entry_tracking_df = pd.merge(puck_carrier_tracking, non_carrier_tracking, how="left", on=['game_id', 'sl_event_id'])
entry_tracking_df = pd.merge(entry_tracking_df, defense_tracking, how="left", on=['game_id', 'sl_event_id'])

In [180]:
entry_tracking_df.to_csv('../results/entry_tracking.csv',index=False)